In [0]:
# =========================================================
# 03_GOLD_ORDERS
# =========================================================


# =========================================================
# 1. GET ACTIVE RUN
# =========================================================

running_runs = spark.sql("""
    SELECT
        run_id,
        batch_id
    FROM workspace.control.etl_run_log
    WHERE pipeline_name = 'orders_pipeline'
      AND status = 'RUNNING'
""").collect()


if len(running_runs) != 1:
    raise ValueError(
        f"Expected exactly 1 RUNNING run, "
        f"found {len(running_runs)}"
    )


run_id = running_runs[0]["run_id"]
batch_id = running_runs[0]["batch_id"]


print(f"Run ID:   {run_id}")
print(f"Batch ID: {batch_id}")


try:

    # =====================================================
    # 2. STAGE DEPENDENCY GUARDRAIL
    #
    # Gold may run ONLY after Silver completed.
    #
    # silver_rows = NULL  -> Silver was not completed
    # silver_rows = 0     -> Silver completed, but inserted 0
    # =====================================================

    run_state = spark.sql(f"""
        SELECT
            landing_rows,
            bronze_rows,
            silver_rows,
            rejected_rows,
            status
        FROM workspace.control.etl_run_log
        WHERE run_id = '{run_id}'
    """).first()


    if run_state is None:
        raise ValueError(
            f"Audit row not found for run_id={run_id}."
        )


    if run_state["silver_rows"] is None:
        raise ValueError(
            f"GOLD BLOCKED: Silver has not completed "
            f"for run_id={run_id}, "
            f"batch_id={batch_id}."
        )


    print("----------------------------------")
    print("STAGE DEPENDENCY CHECK PASSED")
    print("----------------------------------")
    print(f"Landing rows:  {run_state['landing_rows']}")
    print(f"Bronze rows:   {run_state['bronze_rows']}")
    print(f"Silver rows:   {run_state['silver_rows']}")
    print(f"Rejected rows: {run_state['rejected_rows']}")
    print("----------------------------------")


    # =====================================================
    # 3. GOLD MERGE
    # =====================================================

    spark.sql(f"""
        MERGE INTO workspace.gold.orders AS g

        USING (

            SELECT
                order_id,
                customer_id,
                amount,
                currency,
                status,
                order_date,
                last_updated,
                event_id,
                batch_id

            FROM workspace.silver.orders

            WHERE batch_id = '{batch_id}'
              AND dq_overall = 1

            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY
                    last_updated DESC,
                    event_id DESC
            ) = 1

        ) AS s

        ON g.order_id = s.order_id


        WHEN MATCHED
             AND (
                  s.last_updated > g.last_updated

                  OR (
                      s.last_updated = g.last_updated
                      AND s.event_id > g.event_id
                  )
             )

        THEN UPDATE SET

            g.customer_id  = s.customer_id,
            g.amount       = s.amount,
            g.currency     = s.currency,
            g.status       = s.status,
            g.order_date   = s.order_date,
            g.last_updated = s.last_updated,
            g.event_id     = s.event_id,
            g.batch_id     = s.batch_id,
            g.updated_at   = CURRENT_TIMESTAMP()


        WHEN NOT MATCHED

        THEN INSERT (

            order_id,
            customer_id,
            amount,
            currency,
            status,
            order_date,
            last_updated,
            event_id,
            batch_id,
            updated_at

        )

        VALUES (

            s.order_id,
            s.customer_id,
            s.amount,
            s.currency,
            s.status,
            s.order_date,
            s.last_updated,
            s.event_id,
            s.batch_id,
            CURRENT_TIMESTAMP()

        )
    """)


    print("Gold MERGE completed.")


    # =====================================================
    # 4. READ DELTA MERGE METRICS
    # =====================================================

    history = spark.sql("""
        DESCRIBE HISTORY workspace.gold.orders
    """)


    latest_merge = (
        history
        .filter(
            history.operation == "MERGE"
        )
        .orderBy(
            history.version.desc()
        )
        .first()
    )


    if latest_merge is None:
        raise ValueError(
            "No MERGE operation found in Gold history."
        )


    metrics = latest_merge["operationMetrics"]


    gold_inserted = int(
        metrics.get(
            "numTargetRowsInserted",
            0
        )
    )

    gold_updated = int(
        metrics.get(
            "numTargetRowsUpdated",
            0
        )
    )


    print(f"Gold inserted: {gold_inserted}")
    print(f"Gold updated:  {gold_updated}")


    # =====================================================
    # 5. CLOSE AUDIT AS SUCCESS
    # =====================================================

    spark.sql(f"""
        UPDATE workspace.control.etl_run_log

        SET
            gold_inserted = {gold_inserted},
            gold_updated = {gold_updated},
            end_timestamp = CURRENT_TIMESTAMP(),
            status = 'SUCCESS',
            error_message = NULL

        WHERE run_id = '{run_id}'
          AND status = 'RUNNING'
    """)


    print("Audit updated to SUCCESS.")


    # =====================================================
    # 6. FINAL AUDIT OUTPUT
    # =====================================================

    print("----------------------------------")
    print("GOLD COMPLETE")
    print("----------------------------------")
    print(f"Run ID:        {run_id}")
    print(f"Batch ID:      {batch_id}")
    print(f"Gold inserted: {gold_inserted}")
    print(f"Gold updated:  {gold_updated}")
    print("Status:         SUCCESS")
    print("----------------------------------")


    display(
        spark.sql(f"""
            SELECT
                run_id,
                batch_id,
                landing_rows,
                bronze_rows,
                silver_rows,
                rejected_rows,
                gold_inserted,
                gold_updated,
                status,
                start_timestamp,
                end_timestamp,
                error_message
            FROM workspace.control.etl_run_log
            WHERE run_id = '{run_id}'
        """)
    )


except Exception as e:

    # =====================================================
    # 7. FAILURE HANDLING
    # =====================================================

    error_message = str(e)

    print(f"GOLD FAILED: {error_message}")


    safe_error = (
        error_message
        .replace("'", "''")[:2000]
    )


    try:

        spark.sql(f"""
            UPDATE workspace.control.etl_run_log

            SET
                end_timestamp = CURRENT_TIMESTAMP(),
                status = 'FAILED',
                error_message = '{safe_error}'

            WHERE run_id = '{run_id}'
              AND status = 'RUNNING'
        """)

        print("Audit updated to FAILED.")


    except Exception as audit_error:

        print(
            f"WARNING: Failed to update audit row: "
            f"{audit_error}"
        )


    raise

In [0]:
%sql
SELECT
    run_id,
    batch_id,
    landing_rows,
    late_arriving_rows_ingested,
    bronze_rows,
    silver_rows,
    duplicate_event_rows_removed,
    rejected_rows,
    multi_version_order_count,
    gold_inserted,
    gold_updated,
    status,
    error_message
FROM workspace.control.etl_run_log
ORDER BY start_timestamp DESC;